In [77]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler    
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances
from pathlib import Path
import joblib

In [78]:

_cwd = Path.cwd()
if _cwd.name == "notebooks":
    DATA_DIR = (_cwd.parent / "data").resolve()
elif _cwd.name == "backend":
    DATA_DIR = (_cwd / "data").resolve()
else:
    DATA_DIR = (_cwd / "backend" / "data").resolve()
    
features_raw_df = pd.read_parquet(DATA_DIR / "features_raw.parquet")

In [79]:
# Mapping volatility to risk tolerance levels (low, medium, high)
# Lower volatility --> lower risk 
# Higher volatility --> higher risk


risk_thresholds = {
    "low": features_raw_df["volatility"].quantile(0.25),
    "medium": features_raw_df["volatility"].quantile(0.50),
    "high": features_raw_df["volatility"].quantile(0.75)
}

def assign_risk_tolerance_to_stocks(vol):
    if vol <= risk_thresholds["low"]:
        return "low"
    if vol <= risk_thresholds["medium"]:
        return "medium"
    return "high"


In [80]:
# Mapping momentum and divident yield to growth vs. balanced vs. income 
# growth = low dividend yield, high momentum
# balanced = medium dividend yield, medium momentum
# income = high dividend yield, low momentum

# growth oriented --> postivie momentum, low dividend yield

mom_div_thresholds = {
    "growth": {
        "momentum": features_raw_df["momentum"].quantile(0.75), 
        "dividend_yield": features_raw_df["dividend_yield"].quantile(0.25)
        },

        
    "balanced": {
        "momentum": features_raw_df["momentum"].quantile(0.50), 
        "dividend_yield": features_raw_df["dividend_yield"].quantile(0.50)
    },

    "income": {
        "momentum": features_raw_df["momentum"].quantile(0.25), 
        "dividend_yield": features_raw_df["dividend_yield"].quantile(0.75)
    }
}

def assign_growth_profile_to_stocks(mom, div):
        if mom >= mom_div_thresholds["growth"]["momentum"] and div <= mom_div_thresholds["growth"]["dividend_yield"]:
            return "growth"
        if mom <= mom_div_thresholds["income"]["momentum"] and div >= mom_div_thresholds["income"]["dividend_yield"]:
            return "income"
        return "balanced"



In [81]:
# Mapping p/e ratio to investment horizon (1 year vs. 10 year slider)
# high p/e ->  long (5-10 years), you have time for earnings to catch up to valuation
# low p/e -> short (1-2 years), you need the stock to be cheap now

pe_buckets = {
    "short": features_raw_df["pe_ratio"].quantile(0.25),
    "medium": features_raw_df["pe_ratio"].quantile(0.50), 
    "long": features_raw_df["pe_ratio"].quantile(0.75),
}

def year_to_buckets(year):
    if year <= 2:
        return "short"
    if year <= 5: 
        return "medium"
    return "long"
    

def assign_pe_bucket_to_stocks(pe):
    if pe <= features_raw_df["pe_ratio"].quantile(0.25):
        return "short"
    if pe <= features_raw_df["pe_ratio"].quantile(0.50):
        return "medium"
    return "long"


In [ ]:
scaler = joblib.load(DATA_DIR / "scaler.pkl")

# sector map is a df with columns "ticker" and "sector"
sector_map = pd.read_parquet(DATA_DIR / "sector_map.parquet")

# columns: 'momentum', 'volatility', 'pe_ratio', 'dividend_yield' + 10 one-hot sector columns
features_scaled_df = pd.read_parquet(DATA_DIR / "features_scaled.parquet")
features_scaled_df.fillna(features_scaled_df.median(), inplace=True)

# Scaler was fit on these 4 columns in this exact order (see notebook 02, section 9)
QUANT_FEATURES = ["momentum", "volatility", "pe_ratio", "dividend_yield"]


def recommend(user_profile, top_k=5):
    user_sectors = user_profile.get("sectors", [])
    user_risk = user_profile.get("risk_tolerance")
    user_growth = user_profile.get("growth_profile")
    user_horizon = user_profile.get("investment_horizon")

    # filter by sectors first (one-hot columns are 0/1)
    sector_filtered_df = features_scaled_df.copy()
    if user_sectors:
        sector_mask = pd.Series(False, index=sector_filtered_df.index)
        for sector in user_sectors:
            if sector in sector_filtered_df.columns:
                sector_mask |= sector_filtered_df[sector] == 1
        if sector_mask.any():
            sector_filtered_df = sector_filtered_df[sector_mask]

    if sector_filtered_df.empty:
        sector_filtered_df = features_scaled_df.copy()

    n_neighbors = min(top_k, len(sector_filtered_df))
    knn = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean")
    knn.fit(sector_filtered_df[QUANT_FEATURES])

    # Build the user vector in RAW (unscaled) units, with the 4 quant features
    # in the same order the scaler was fit on.
    user_raw = {
        "momentum": mom_div_thresholds[user_growth]["momentum"] if user_growth else 0,
        "volatility": risk_thresholds[user_risk] if user_risk else 0,
        "pe_ratio": pe_buckets[user_horizon] if user_horizon else 0,
        "dividend_yield": mom_div_thresholds[user_growth]["dividend_yield"] if user_growth else 0,
    }
    user_df = pd.DataFrame([user_raw], columns=QUANT_FEATURES)
    user_scaled = scaler.transform(user_df)

    distances, indices = knn.kneighbors(user_scaled)
    return sector_filtered_df.index[indices[0]].tolist()


In [83]:
test_profile = {
    "sectors": ["Technology", "Healthcare"],
    "risk_tolerance": "medium",
    "growth_profile": "balanced",
    "investment_horizon": "medium"
}

print(recommend(test_profile, top_k = 5))

ValueError: The feature names should match those that were passed during fit.
Feature names must be in the same order as they were in fit.


In [ ]:
features_scaled_df

,momentum,volatility,pe_ratio,dividend_yield,Consumer Discretionary,Consumer Staples,Energy,Financials,Healthcare,Industrials,Materials,Real Estate,Technology,Utilities
AAPL,0.609620,-0.606579,0.118494,-1.073067,0,0,0,0,0,0,0,0,1,0
ABBV,-0.337890,-0.427894,2.857330,0.471884,0,0,0,0,1,0,0,0,0,0
ABNB,0.144372,-0.011825,0.119779,-1.270075,1,0,0,0,0,0,0,0,0,0
ADC,-0.222836,-1.224984,0.395001,0.917743,0,0,0,0,0,0,0,1,0,0
AEM,-0.141257,1.249201,-0.590729,-0.746450,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WFC,-0.318990,-0.343505,-0.819259,-0.103584,0,0,0,1,0,0,0,0,0,0
WMB,-0.453447,-0.658931,0.017121,0.160820,0,0,1,0,0,0,0,0,0,0
WMT,0.072807,-0.725744,0.659377,-0.876060,0,1,0,0,0,0,0,0,0,0
WSM,-0.150250,0.379277,-0.454494,-0.393911,1,0,0,0,0,0,0,0,0,0
